# Financial Fraud Detection (Colab)

Self-contained notebook version of this repo's `fraud_detection` package: generates a synthetic, labeled transaction dataset, trains a `RandomForestClassifier` fraud model, evaluates it, and scores example transactions.

Run all cells top to bottom (`Runtime -> Run all`).

In [ ]:
!pip install -q scikit-learn pandas numpy joblib

## 1. Synthetic data generation

Mostly legitimate transactions plus a small fraction of fraud with distinct signatures: odd hours, large amounts, location jumps, high velocity.

In [ ]:
import numpy as np
import pandas as pd

FEATURE_COLUMNS = [
    "amount",
    "hour",
    "distance_from_home_km",
    "distance_from_last_txn_km",
    "txns_last_hour",
    "is_foreign_country",
    "is_new_merchant",
    "avg_amount_ratio",
]


def generate_transactions(n_samples=50_000, fraud_ratio=0.015, random_state=42):
    rng = np.random.default_rng(random_state)
    n_fraud = int(n_samples * fraud_ratio)
    n_legit = n_samples - n_fraud

    legit = pd.DataFrame({
        "amount": rng.gamma(shape=2.0, scale=40, size=n_legit),
        "hour": rng.normal(loc=14, scale=4, size=n_legit) % 24,
        "distance_from_home_km": rng.exponential(scale=5, size=n_legit),
        "distance_from_last_txn_km": rng.exponential(scale=3, size=n_legit),
        "txns_last_hour": rng.poisson(lam=0.5, size=n_legit),
        "is_foreign_country": rng.binomial(1, 0.02, size=n_legit),
        "is_new_merchant": rng.binomial(1, 0.1, size=n_legit),
        "avg_amount_ratio": rng.normal(loc=1.0, scale=0.3, size=n_legit).clip(0.1),
    })
    legit["is_fraud"] = 0

    fraud = pd.DataFrame({
        "amount": rng.gamma(shape=3.0, scale=250, size=n_fraud),
        "hour": rng.normal(loc=3, scale=3, size=n_fraud) % 24,
        "distance_from_home_km": rng.exponential(scale=400, size=n_fraud),
        "distance_from_last_txn_km": rng.exponential(scale=300, size=n_fraud),
        "txns_last_hour": rng.poisson(lam=4, size=n_fraud),
        "is_foreign_country": rng.binomial(1, 0.55, size=n_fraud),
        "is_new_merchant": rng.binomial(1, 0.7, size=n_fraud),
        "avg_amount_ratio": rng.normal(loc=6.0, scale=3.0, size=n_fraud).clip(0.5),
    })
    fraud["is_fraud"] = 1

    df = pd.concat([legit, fraud], ignore_index=True)
    df = df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    df["amount"] = df["amount"].round(2)
    df["hour"] = df["hour"].round(2)
    return df


df = generate_transactions()
print(df.shape)
df.head()

## 2. Train the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = df[FEATURE_COLUMNS]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    )),
])

pipeline.fit(X_train, y_train)

proba = pipeline.predict_proba(X_test)[:, 1]
preds = pipeline.predict(X_test)

print(f"ROC AUC:           {roc_auc_score(y_test, proba):.4f}")
print(f"Average precision: {average_precision_score(y_test, proba):.4f}")
print(classification_report(y_test, preds, digits=3))

## 3. Score example transactions

In [ ]:
examples = pd.DataFrame([
    {  # ordinary daytime purchase
        "amount": 25.0, "hour": 13.0, "distance_from_home_km": 1.5,
        "distance_from_last_txn_km": 0.5, "txns_last_hour": 0,
        "is_foreign_country": 0, "is_new_merchant": 0, "avg_amount_ratio": 1.0,
    },
    {  # suspicious late-night, high-value, foreign transaction
        "amount": 4000.0, "hour": 3.0, "distance_from_home_km": 900.0,
        "distance_from_last_txn_km": 700.0, "txns_last_hour": 6,
        "is_foreign_country": 1, "is_new_merchant": 1, "avg_amount_ratio": 9.0,
    },
])[FEATURE_COLUMNS]

examples["fraud_probability"] = pipeline.predict_proba(examples)[:, 1]
examples

## 4. Save the model (optional)

Download `model.joblib` afterwards from the Colab file browser if you want to reuse it with `fraud_detection/api.py` from the repo.

In [ ]:
import joblib

joblib.dump(pipeline, "model.joblib")
print("Saved model.joblib")